# CheXzero Baseline — Google Colab

**Project:** Vision-Language Models in Radiology  
**Repo:** https://github.com/SinaDns/radiology-vision-language-models

This notebook:
1. Clones the repository and installs dependencies
2. Downloads the IU X-Ray dataset
3. Runs import / forward-pass smoke tests
4. Trains CheXzero contrastively on IU X-Ray
5. Evaluates zero-shot classification on NIH ChestX-ray14

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* before running.
>
> **NIH-14 evaluation (Section 5) requires a manual download step** — see that section for instructions.

---
## 1. Environment Setup

In [ ]:
import os, subprocess, sys

# ── Clone / pull repo ────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/SinaDns/radiology-vision-language-models.git"
REPO_DIR  = "/content/radiology-vision-language-models"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# ── Install Python dependencies ──────────────────────────────────────────────
# Colab already ships with torch/torchvision; we upgrade them if needed
# and install the remaining packages from requirements.txt.
!pip install -q -r requirements.txt

In [ ]:
# ── Logging + GPU check ───────────────────────────────────────────────────────
import logging, torch

# Configure root logger so all src.* module logs appear in Colab output.
# Each log line shows timestamp, level, and which module emitted the message,
# which makes remote debugging from logs much easier.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S',
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

---
## 2. Download IU X-Ray Dataset

~7,470 PNG images + ~3,955 XML reports from the NIH OpenI collection.  
Total size ≈ 1.4 GB.

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("/content/radiology-vision-language-models/data/iu_xray")
DATA_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_TGZ  = DATA_DIR / "NLMCXR_png.tgz"
REPORT_TGZ = DATA_DIR / "NLMCXR_reports.tgz"

# Download (skip if already present)
if not IMAGE_TGZ.exists():
    print("Downloading images (~1.3 GB)…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz" \
        -O {IMAGE_TGZ}
else:
    print("Image archive already present.")

if not REPORT_TGZ.exists():
    print("Downloading reports…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz" \
        -O {REPORT_TGZ}
else:
    print("Report archive already present.")

In [ ]:
# ── Extract ──────────────────────────────────────────────────────────────────
IMAGES_DIR  = DATA_DIR / "images"
REPORTS_DIR = DATA_DIR / "reports"
IMAGES_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

import tarfile

if not any(IMAGES_DIR.glob("*.png")):
    print("Extracting images…")
    with tarfile.open(IMAGE_TGZ, "r:gz") as tar:
        for member in tar.getmembers():
            member.name = Path(member.name).name  # flatten directory
            tar.extract(member, IMAGES_DIR)
    print("Done.")

if not any(REPORTS_DIR.glob("*.xml")):
    print("Extracting reports…")
    with tarfile.open(REPORT_TGZ, "r:gz") as tar:
        for member in tar.getmembers():
            member.name = Path(member.name).name
            tar.extract(member, REPORTS_DIR)
    print("Done.")

n_images  = len(list(IMAGES_DIR.glob("*.png")))
n_reports = len(list(REPORTS_DIR.glob("*.xml")))
print(f"PNG images : {n_images}   (expected ~7,470)")
print(f"XML reports: {n_reports}  (expected ~3,955)")

---
## 3. Smoke Tests (no data required)

In [ ]:
# ── Data loader import ───────────────────────────────────────────────────────
from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
print("Data loaders OK")

In [ ]:
# ── Model forward pass ───────────────────────────────────────────────────────
import torch
from src.models.chexzero import CheXzero

# This downloads BioClinicalBERT weights (~400 MB) on the first run.
model = CheXzero(embed_dim=512)

imgs  = torch.randn(2, 3, 320, 320)
ids   = torch.randint(0, 1000, (2, 64))
mask  = torch.ones(2, 64, dtype=torch.long)

logits_i, logits_t = model(imgs, ids, mask)
print("Forward pass OK — logits shape:", logits_i.shape)  # expected (2, 2)

In [ ]:
# ── Config loading ───────────────────────────────────────────────────────────
from src.utils.config import load_config
config = load_config("experiments/configs/chexzero.yaml")
print(config)

In [ ]:
# ── Loss function sanity check ───────────────────────────────────────────────
from src.training.losses import clip_loss
B = 4
logits = torch.eye(B) * 10  # perfect alignment
loss = clip_loss(logits, logits.T)
print(f"clip_loss (perfect alignment, B={B}): {loss.item():.4f}  (should be near 0)")

---
## 4. Training

Trains CheXzero contrastively on IU X-Ray.  
Checkpoints are saved to `experiments/results/checkpoints/chexzero/`.

In [ ]:
# ── Config overrides for Colab ───────────────────────────────────────────────
# Colab T4 has 15 GB VRAM. batch_size=64 is safe; increase to 128 if using A100.

from src.utils.config import load_config
import copy

config = load_config("experiments/configs/chexzero.yaml")

# Point to Colab data paths
config["paths"]["iu_xray_dir"]     = "data/iu_xray/"
config["paths"]["nih14_dir"]       = "data/nih_chestxray14/"
config["paths"]["checkpoint_dir"]  = "experiments/results/checkpoints/chexzero/"
config["paths"]["log_dir"]         = "experiments/results/logs/"

# Reduce batch size for T4 (15 GB); set to 128 for A100
config["training"]["batch_size"] = 64

# Reduce workers to avoid Colab shared-memory issues
config["data"]["num_workers"] = 2

print("Config ready:", config)

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
from src.models.chexzero import CheXzero
from src.training.contrastive import ContrastiveTrainer
from src.utils.logging_utils import setup_logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

image_size   = config["data"]["image_size"]
val_fraction = config["data"]["val_split"]
iu_xray_dir  = config["paths"]["iu_xray_dir"]

train_dataset = IUXrayDataset(
    data_dir=iu_xray_dir,
    split="train",
    val_fraction=val_fraction,
    transform=get_train_transforms(image_size),
)
val_dataset = IUXrayDataset(
    data_dir=iu_xray_dir,
    split="val",
    val_fraction=val_fraction,
    transform=get_val_transforms(image_size),
)
print(f"Train: {len(train_dataset)} samples  |  Val: {len(val_dataset)} samples")

batch_size   = config["training"]["batch_size"]
num_workers  = config["data"]["num_workers"]

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

model  = CheXzero(embed_dim=config["model"]["embed_dim"])
logger = setup_logger(config["paths"]["log_dir"], config["logging"]["run_name"])

trainer = ContrastiveTrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    logger=logger,
)

print("Trainer ready.")

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────────
# Set resume_path to continue from a previous checkpoint, e.g.:
#   resume_path = "experiments/results/checkpoints/chexzero/latest.pt"

trainer.train(resume_path=None)

In [ ]:
# ── (Optional) Save checkpoint to Google Drive ───────────────────────────────
# Uncomment to persist the best checkpoint across Colab sessions.

# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy(
#     "experiments/results/checkpoints/chexzero/best.pt",
#     "/content/drive/MyDrive/chexzero_best.pt",
# )
# print("Checkpoint saved to Google Drive.")

---
## 5. Zero-Shot Evaluation on NIH ChestX-ray14

**Manual download required** — NIH-14 is not publicly accessible via `wget`.

Steps:
1. Go to https://nihcc.app.box.com/v/ChestXray-NIHCC in your browser
2. Download `images_001.tar.gz` … `images_012.tar.gz`, `Data_Entry_2017.csv`, `test_list.txt`
3. Upload them to Google Drive and mount Drive below, **or** upload directly to Colab's `/content/` folder
4. Update `NIH14_ZIPS_DIR` in the next cell and run

In [ ]:
# ── Mount Google Drive (if NIH-14 files are stored there) ────────────────────
# from google.colab import drive
# drive.mount("/content/drive")

# Path where you uploaded the NIH-14 files (zips + CSVs)
NIH14_ZIPS_DIR = "/content/drive/MyDrive/nih_chestxray14"   # ← update this

NIH14_DIR = "/content/radiology-vision-language-models/data/nih_chestxray14"
import os
os.makedirs(NIH14_DIR + "/images", exist_ok=True)
print("NIH-14 dir:", NIH14_DIR)

In [ ]:
# ── Extract NIH-14 image archives ────────────────────────────────────────────
import tarfile, glob
from pathlib import Path

images_dir = Path(NIH14_DIR) / "images"
archives   = sorted(glob.glob(f"{NIH14_ZIPS_DIR}/images_*.tar.gz"))

if not archives:
    print("No image archives found at", NIH14_ZIPS_DIR)
    print("Update NIH14_ZIPS_DIR and re-run this cell.")
else:
    for arc in archives:
        print(f"Extracting {Path(arc).name}…")
        with tarfile.open(arc, "r:gz") as tar:
            tar.extractall(images_dir)
    print(f"Total images: {len(list(images_dir.rglob('*.png')))}  (expected ~112,120)")

# Copy metadata
import shutil
for fname in ["Data_Entry_2017.csv", "test_list.txt", "train_val_list.txt"]:
    src = Path(NIH14_ZIPS_DIR) / fname
    if src.exists():
        shutil.copy(src, Path(NIH14_DIR) / fname)
        print(f"Copied {fname}")

In [ ]:
# ── Run zero-shot evaluation ─────────────────────────────────────────────────
import torch
import numpy as np
from torch.utils.data import DataLoader

from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset, NIH14_LABELS
from src.data_loaders.transforms import get_val_transforms
from src.evaluation.zero_shot import compute_zero_shot_scores
from src.evaluation.metrics import compute_auroc, compute_auprc, classification_report
from src.models.chexzero import CheXzero
from src.utils.config import load_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained model
CHECKPOINT = "experiments/results/checkpoints/chexzero/best.pt"
# Uncomment to load from Drive instead:
# CHECKPOINT = "/content/drive/MyDrive/chexzero_best.pt"

config = load_config("experiments/configs/chexzero.yaml")
model  = CheXzero(embed_dim=config["model"]["embed_dim"])
ckpt   = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Checkpoint loaded (epoch {ckpt.get('epoch', '?')}).")

# Dataset
test_dataset = NIHChestXray14Dataset(
    data_dir=NIH14_DIR,
    split="test",
    split_csv=f"{NIH14_DIR}/test_list.txt",
    transform=get_val_transforms(config["data"]["image_size"]),
)
print(f"Test samples: {len(test_dataset)}")

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Zero-shot scores
print("Computing zero-shot scores…")
scores_dict = compute_zero_shot_scores(
    model=model,
    image_loader=test_loader,
    labels=NIH14_LABELS,
    device=device,
)

# Ground-truth labels
all_labels = {l: [] for l in NIH14_LABELS}
for batch in test_loader:
    gt = batch["labels"].numpy()
    for i, label in enumerate(NIH14_LABELS):
        all_labels[label].append(gt[:, i])
labels_dict = {l: np.concatenate(all_labels[l]) for l in NIH14_LABELS}

# Metrics
auroc_results = compute_auroc(scores_dict, labels_dict)
auprc_results = compute_auprc(scores_dict, labels_dict)

score_matrix = np.stack([scores_dict[l] for l in NIH14_LABELS], axis=1)
label_matrix = np.stack([labels_dict[l] for l in NIH14_LABELS], axis=1)

print("\n=== Zero-Shot Classification Results (NIH ChestX-ray14) ===")
print(classification_report(score_matrix, label_matrix, label_names=NIH14_LABELS))
print(f"\nMean AUROC : {auroc_results['mean_auroc']:.4f}")
print(f"Mean AUPRC : {auprc_results['mean_auprc']:.4f}")
print("\nTarget: mean AUROC ≥ 0.75 (zero-shot, no NIH-14 training data)")

---
## 6. Results Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels_to_plot = [l for l in NIH14_LABELS if l in auroc_results]
auroc_vals     = [auroc_results[l] for l in labels_to_plot]
mean_auroc     = auroc_results["mean_auroc"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_to_plot, auroc_vals, color="steelblue")
ax.axvline(mean_auroc, color="red", linestyle="--", label=f"Mean AUROC = {mean_auroc:.3f}")
ax.axvline(0.75, color="orange", linestyle=":", label="Target (0.75)")
ax.set_xlabel("AUROC")
ax.set_title("CheXzero Zero-Shot AUROC on NIH ChestX-ray14")
ax.set_xlim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig("experiments/results/chexzero_auroc.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/chexzero_auroc.png")